This is an attempt to reimplement the FCOS model based on the paper by [Tian et al., 2019, ICCV 2019.](https://openaccess.thecvf.com/content_ICCV_2019/papers/Tian_FCOS_Fully_Convolutional_One-Stage_Object_Detection_ICCV_2019_paper.pdf)

FCOS is designed to be anchor-box free and proposal free.
- FCOS does not use anchor-box related computation such as calculating overlap during training, avoids hyper-parameters related to anchor boxes. FCOS encourages to rethink the need to anchor boxes.

Read this [article](https://sh-tsang.medium.com/review-fcos-fully-convolutional-one-stage-object-detection-90d57274b19f) for more details.



According to the Experiments section
1. COCO trainval35k split (115K images) for training, minival split (5K images) as validation. test_dev split (20K images) for evaluation.
2. ResNet-50 is used as backbone using pre-trained weights of ImageNet.
3. Hyperparameters use RetinaNet.
4. Network trained with Stochastic Gradient Descent (SGD) for 90K iterations
5. with initial learning rate of 0.01 and mini-batch of 16 images.
6. Learning rate reduce by a factor of 10 at iteration 60K and 80K; 0.01(<60K), 0.001 (60K-90K), 0.0001 (90K-115K).
7. Input images are resized to have the shorter side being 800 and longer being less or equal to 1333.

Inference
1. Forward the input image through the network and <u>obtain predicted bounding box with a predicted class</u>
2. Post-processing is the same with RetinaNet, directly use similar post-processing hyperparameters of RetinaNet.
- Same size of input image as in training.

FCOS: Resolving issue of FCN-bassed having low recall rate and ambiguous samples
- Major concern for FCN-based detector

### Import Libraries

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# To impost COCO trainval35k split (115K images)
# minival split (5K images)
import tensorflow_datasets as tfds

### Step 1: Backbone using ResNet50

The paper mentioned

In [ ]:
def getBackbone():
  # Load ResNet50 model pretrained on ImageNet
  resnet = tf.keras.applications.ResNet50(weights="imagenet", include_top=False)

  # Freeze the weights of the backbone
  for layer in resnet.layers:
    layer.trainable = False

  return resnet

###Step 2: Feature Pyramid Network (FPN)
There are several operations like lateral connections and top-down path

In [ ]:
class FPN (tf.keras.layers.Layer):
  def __init__(self, input_channels_list=[256, 512, 1024, 2048], output_channel=256, **kwargs):
    super(FPN, self).__init__(**kwargs)
    self.input_channels_list = input_channels_list
    self.output_channel = output_channel

    # Lateral layers (downsampling)
    self.lateral_convs = [layers.Conv2D(outpu_channel, kernel_size=1,activation='relu') for _ in range(len(input_channels_list))]

    # Top-down path with convolutions
    self.upsample_convs = []
    for idx in range(len(input_channels_list) - 1):
      self.upsample_convs.append(layers.Conv2D(output_channel, kernel_size=3, strides=2, padding='same', activation='relu'))

    # Final output layers
    self.output_convs = [layers.Conv2D(output_channel, kernel_size=3, padding='same', activation='relu') for _ in range(len(input_channels_list))]

    def call(self, inputs):
      features = [lateral_conv(inputs[i]) for i, lateral_conv in enumerate(self.lateral_convs)]

      # Top-down path
      top_down = features[-1]
      for i in reversed(range(len(features) - 1)):
        top_down = self.upsample_convs[i](top_down)
        top_down = layers.Concatenate()([top_down, features[i]])
        top_down = self.output_convs[i][top_down]

      return top_down

###Step 3: Prediction Layers for FCOS
Prediction layers for bounding box coordinates, class scores, and center-ness scores.

In [ ]:
class FCOSPredictionHead(tf.keras.layers.Layer):
  def __init__(self, output_channels=256, num_classes=80, **kwargs):
    super(FCOSPredictionHead, self).__init__(**kwargs)
    self.output_channels = output_channels
    self.num_classes = num_classes

    # Bounding box regression head
    self.bbox_head = layers.Conv2D(4 * (num_classes + 3), kernel_size=3, padding='same', activation='sigmoid')

    # Class prediction head
    self.cls_head = layers.Conv2D(num_classes, kernel_size=3, padding='same', activation='softmax')

    # Center-ness score head
    self.center_score_head = layers.Conv2D(1, kernel_size=3, padding='same', activation='sigmoid')

    def call(self,features):
      bbox_pred = self.bbox_head(features)
      cls_pred = self.cls_head(features)
      center_score = self.center_score_head(features)

      return bbox_pred, cls_pred, center_score

###Step 4: FCOS

In [ ]:
class FCOS(tf.keras.Model):
  def __init__(self, **kwargs):
    super(FCOS, self).__init__(**kwargs)

    # Backbone (ResNet50)
    self.backbone = getBackbone()

    # FPN
    self.fpn = FPN()

    # Prediciton layers
    self.prediction_head = FCOSPredictionHead()

  def call(self, inputs):
    # Pass through backbone
    features = list(self.backbone(inputs).values())

    # FPN on the last featuer map (usually the one with the highest resolution)
    top_down_feature = features[-1]

    # FPN with multi-scale feature maps
    for i in range(len(features) - 2, -1, -1):
      top_down_feature = self.fpn.layers[i + len(self.fpn.upsample_convs)](top_down_feature)
      top_down_feature = layers.Concatenate()([top_down_feature, features-[i]])

    # Get predictions
    bbox_pred, cls_pred, center_score = self.prediction_head(top_down_feature)

    return bbox_pred, cls_pred, center_score